In [3]:
import torch
from torch import nn
from torch.nn import functional as F
import math

In [14]:
class Attention(nn.Module):
    def __init__(self, dim):
        super(Attention, self).__init__()
        self.dim = dim
        # fc để biến input thành Q, K, V
        self.fc_input = nn.LazyLinear(out_features=dim)
    
    def forward(self, q_input, k_input, v_input, mask=None):
        # Input shape: [Batch_size, Sequence_Length, d_model]
        # T_q = T_k = T_v
        Q = self.fc_input(q_input) # Shape: [B, T_q, d_model]
        K = self.fc_input(k_input) # Shape: [B, T_k, d_model]
        V = self.fc_input(v_input) # Shape: [B, T_v, d_model]

        # transpose(-2, -1) sẽ lật 2 chiều cuối của K từ [B, T_k, d_model] thành [B, d_model, T_k]
        # Phép nhân: [B, T_q, d_model] x [B, d_model, T_k] -> Kết quả: [B, T_q, T_k]
        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(self.dim)

        if mask is not None:
            # Che đi các vị trí không muốn chú ý tới (đổi thành -vô cực)
            scores = scores.masked_fill(mask == 0, -1e9)
        # softmax cho scores
        attn_weights = torch.softmax(scores, dim=-1)

        # [B, T_q, T_k] x [B, T_k, d_model] -> Kết quả: [B, T_q, d_model]
        output = torch.matmul(attn_weights, V)

        return output, attn_weights

In [12]:
att = Attention(16)
X = torch.rand(2, 5, 16)
output, attn_weights = att(q_input=X, k_input=X, v_input=X, mask=False)
print(X.shape)
print(output.shape)
print(attn_weights)

torch.Size([2, 5, 16])
torch.Size([2, 5, 16])
tensor([[[0.2175, 0.1959, 0.1903, 0.1968, 0.1994],
         [0.2036, 0.2090, 0.1973, 0.1974, 0.1928],
         [0.2015, 0.2010, 0.2006, 0.1966, 0.2002],
         [0.2012, 0.1942, 0.1898, 0.2115, 0.2034],
         [0.2039, 0.1896, 0.1934, 0.2034, 0.2096]],

        [[0.2191, 0.1885, 0.1980, 0.2089, 0.1856],
         [0.2007, 0.2071, 0.2001, 0.2083, 0.1838],
         [0.1993, 0.1892, 0.2282, 0.2035, 0.1799],
         [0.1986, 0.1861, 0.1922, 0.2414, 0.1818],
         [0.2026, 0.1885, 0.1951, 0.2087, 0.2051]]],
       grad_fn=<SoftmaxBackward0>)


In [15]:
mask = torch.tril(torch.ones(5, 5))

mask_att = Attention(dim=16)
ouputs, attn_weights = mask_att(X, X, X, mask=mask)
print(attn_weights)

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5011, 0.4989, 0.0000, 0.0000, 0.0000],
         [0.3448, 0.3070, 0.3482, 0.0000, 0.0000],
         [0.2555, 0.2419, 0.2387, 0.2639, 0.0000],
         [0.2086, 0.1876, 0.1963, 0.1988, 0.2088]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4493, 0.5507, 0.0000, 0.0000, 0.0000],
         [0.3031, 0.3145, 0.3824, 0.0000, 0.0000],
         [0.2153, 0.2325, 0.2596, 0.2926, 0.0000],
         [0.1682, 0.1932, 0.1957, 0.2205, 0.2225]]],
       grad_fn=<SoftmaxBackward0>)
